# Analisis Bisnis Resale Sneaker: Defile de Mode (2022–2024)

**Penulis:** Baihaqsani  
**Tools:** Python, Pandas, Matplotlib  
**Data:** 1.038 baris transaksi dari bisnis resale sneaker selama 3 tahun

---

## Konteks Bisnis

Defile de Mode adalah bisnis resale sneaker yang beroperasi sejak 2019 hingga sekarang, menjual melalui berbagai platform marketplace termasuk Alias, Kick Avenue, Direct Transfer, Tokopedia, Shopee, Ox Street, dan Novelship. Modal bersumber dari dana pribadi maupun investor eksternal, dengan pembagian keuntungan sesuai kesepakatan.

## Pertanyaan yang Dijawab Analisis Ini

1. **Platform penjualan mana yang paling efisien secara modal?** (margin per channel)
2. **Brand mana yang menghasilkan return terbaik per rupiah modal?** (margin per brand)
3. **Berapa banyak modal yang tertahan di stok yang belum terjual?** (analisis dead stock)

## Temuan Utama (Ringkasan)

- **Total profit bersih (2022–2024):** Rp 364.278.382 dari 879 transaksi terjual
- **Alias mengungguli Kick Avenue** sebesar 6,5 poin margin pada volume sebanding (~300 transaksi masing-masing)
- **Adidas menghasilkan margin 35%** vs 23% Nike, meskipun Nike memiliki 5x lebih banyak transaksi
- **Rp 288,7 juta (16,5% dari total modal) tertahan di 159 barang yang belum terjual**, semuanya nyangkut lebih dari 1 tahun

---

## 1. Memuat & Menstandardisasi Data

Data transaksi tersimpan dalam tiga file CSV terpisah (satu per tahun). Setiap file memiliki posisi baris header yang berbeda karena format spreadsheet asli yang tidak konsisten. Nama kolom juga berbeda antar tahun dan distandarisasi sebelum digabungkan.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Muat data tiap tahun — posisi baris header berbeda per file (diverifikasi manual)
df_2022 = pd.read_csv("/content/DATA_TRANSAKSI_DEFILE_DE_MODE_2022-2024_-_DATA_TRANSAKSI_DEFILE_2022.csv", header=4)
df_2023 = pd.read_csv("/content/DATA_TRANSAKSI_DEFILE_DE_MODE_2022-2024_-_DATA_TRANSAKSI_DEFILE_2023.csv", header=3)
df_2024 = pd.read_csv("/content/DATA_TRANSAKSI_DEFILE_DE_MODE_2022-2024_-_DATA_TRANSAKSI_DEFILE_2024.csv", header=3)

# Standardisasi nama kolom antar tahun
df_2022 = df_2022.rename(columns={"Nama Investor": "Investor"})
df_2024 = df_2024.rename(columns={"Harga Penjualan": "Penjualan"})

# Gabungkan menjadi satu DataFrame
df_all = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)

print(f"Total baris: {len(df_all)}")
df_all.info()

## 2. Pembersihan Data

Empat masalah teridentifikasi dan diselesaikan:
1. **Kolom uang tersimpan sebagai teks** — tanda koma pada angka mencegah konversi numerik
2. **Nama channel tidak konsisten** — typo dan variasi spasi di berbagai entri
3. **Nama brand tidak konsisten** — masalah kapitalisasi dan satu typo ("Addidas")
4. **21 baris modal nol** — barang yang menang dari raffle, dicatat tanpa biaya pembelian

In [ ]:
# Konversi kolom uang dari teks ke numerik
# Tanda koma sebagai pemisah ribuan harus dihapus sebelum konversi
# errors='coerce' mengubah nilai yang tidak bisa diparse (mis. '-') menjadi NaN
money_cols = ["Modal", "Penjualan", "Pembagian Untung", "Untung Bersih"]

for col in money_cols:
    df_all[col] = pd.to_numeric(df_all[col].astype(str).str.replace(",", ""), errors="coerce")

# Bersihkan nama channel: perbaiki spasi, kapitalisasi, dan typo
df_all["Platform Penjualan"] = df_all["Platform Penjualan"].str.strip().str.title()
df_all["Platform Penjualan"] = df_all["Platform Penjualan"].replace({
    "Direct Teansfer": "Direct Transfer",
    "Kick Avneue": "Kick Avenue",
    "Oxstreet": "Ox Street"
})

# Bersihkan nama brand: perbaiki spasi, kapitalisasi, dan typo
df_all["Brand"] = df_all["Brand"].str.strip().str.title()
df_all["Brand"] = df_all["Brand"].replace({"Addidas": "Adidas"})

# Atur format tampilan untuk keterbacaan (tidak mengubah data)
pd.options.display.float_format = '{:,.0f}'.format

print("Channel setelah pembersihan:")
print(df_all["Platform Penjualan"].value_counts())
print("\nBrand setelah pembersihan:")
print(df_all["Brand"].value_counts())

In [ ]:
# Pisahkan barang terjual vs belum terjual
# Kolom 'Penjualan' kosong = barang belum terjual (dead stock)
terjual = df_all[df_all["Penjualan"].notna()]    # barang terjual
dead_stock = df_all[df_all["Penjualan"].isna()]  # barang belum terjual

# Pisahkan item modal nol (kemenangan raffle) untuk analisis margin yang bersih
gratis = terjual[terjual["Modal"] == 0]      # 21 barang raffle
bermodal = terjual[terjual["Modal"] > 0]     # 858 barang dengan modal

# Verifikasi
print(f"Terjual: {len(terjual)} | Dead stock: {len(dead_stock)} | Total: {len(terjual) + len(dead_stock)}")
print(f"Bermodal: {len(bermodal)} | Raffle (modal nol): {len(gratis)}")

**Catatan tentang barang modal nol:** 21 barang diperoleh dari raffle sneaker tanpa biaya pembelian. Barang ini dikecualikan dari perhitungan margin (margin = profit ÷ modal — tidak terdefinisi saat modal = 0) namun dilaporkan terpisah karena merepresentasikan profit murni tanpa risiko modal.

## 3. Ringkasan Profit Keseluruhan

In [ ]:
total_profit = terjual["Untung Bersih"].sum()
raffle_profit = gratis["Untung Bersih"].sum()
capital_profit = bermodal["Untung Bersih"].sum()

print(f"Total profit bersih (semua transaksi):  Rp {total_profit:,.0f}")
print(f"  - Dari barang bermodal:               Rp {capital_profit:,.0f}")
print(f"  - Dari barang raffle (modal nol):     Rp {raffle_profit:,.0f} ({raffle_profit/total_profit*100:.1f}% dari total)")

Barang raffle hanya berkontribusi 0,9% dari total profit dari 21 transaksi. Meskipun barang ini tidak memiliki risiko modal (margin 100% secara definisi), rata-rata profit per transaksinya (Rp 162.723) **2,6x lebih rendah** dibanding barang bermodal (Rp 421.226), menunjukkan bahwa pembelian yang dipilih secara sengaja konsisten menghasilkan return absolut lebih besar daripada kemenangan raffle acak.

## 4. Analisis Channel Penjualan

Margin dihitung sebagai: **(total profit ÷ total modal yang diinvestasikan) × 100**

Hanya item dengan modal > 0 yang dimasukkan untuk memastikan margin mencerminkan efisiensi modal yang sesungguhnya.

In [ ]:
channel = bermodal.groupby("Platform Penjualan").agg(
    total_modal=("Modal", "sum"),
    total_untung=("Untung Bersih", "sum")
)
channel["margin_pct"] = channel["total_untung"] / channel["total_modal"] * 100
channel["n_transaksi"] = bermodal.groupby("Platform Penjualan")["Untung Bersih"].count()
channel.sort_values("margin_pct", ascending=False)

In [ ]:
channel_plot = channel.sort_values("margin_pct", ascending=True)

warna_channel = ["#4C72B0" if c != channel_plot["margin_pct"].idxmax()
                 else "#2CA02C" for c in channel_plot.index]

plt.figure(figsize=(9, 6))
bars = plt.barh(channel_plot.index, channel_plot["margin_pct"], color=warna_channel)

for bar, (idx, row) in zip(bars, channel_plot.iterrows()):
    lebar = bar.get_width()
    tengah_y = bar.get_y() + bar.get_height() / 2
    plt.text(lebar + 0.5, tengah_y, f"{lebar:.1f}%", va="center", fontsize=10)
    if lebar > 8:
        plt.text(lebar / 2, tengah_y, f"n={int(row['n_transaksi'])}",
                 va="center", ha="center", fontsize=9, color="white")

plt.title("Margin Keuntungan per Channel Penjualan", fontsize=13)
plt.xlabel("Margin (%)")
plt.ylabel("Channel")
plt.xlim(0, channel_plot["margin_pct"].max() + 8)
plt.tight_layout()
plt.show()

**Temuan:**

Novelship (54,2%) dan Ox Street (49,9%) menunjukkan margin tertinggi, namun dengan masing-masing hanya 2 dan 5 transaksi, angka ini tidak dapat diandalkan secara statistik — satu transaksi outlier bisa mendistorsi sampel kecil secara dramatis.

Perbandingan paling valid adalah **Alias vs Kick Avenue** — keduanya memiliki volume transaksi sebanding (~300 masing-masing), menjadikan ini perbandingan yang adil:
- **Alias: margin 28,5%** (n=301)
- **Kick Avenue: margin 22,0%** (n=307)

**Rekomendasi:** Mengalihkan sebagian alokasi modal dari Kick Avenue ke Alias berpotensi meningkatkan efisiensi modal keseluruhan sekitar 6,5 poin persentase tanpa mengorbankan volume transaksi.

## 5. Analisis Brand

Metodologi sama dengan analisis channel. Item modal nol dikecualikan.

In [ ]:
margin_brand_bersih = bermodal.groupby("Brand").agg(
    total_modal=("Modal", "sum"),
    total_untung=("Untung Bersih", "sum")
)
margin_brand_bersih["margin_pct"] = (
    margin_brand_bersih["total_untung"] / margin_brand_bersih["total_modal"] * 100
)
margin_brand_bersih["n_transaksi"] = bermodal.groupby("Brand")["Untung Bersih"].count()
margin_brand_bersih.sort_values("margin_pct", ascending=False)

In [ ]:
brand_plot = margin_brand_bersih.copy()
brand_plot = brand_plot.sort_values("margin_pct", ascending=True)

warna_brand = ["#4C72B0" if c != brand_plot["margin_pct"].idxmax()
               else "#2CA02C" for c in brand_plot.index]

plt.figure(figsize=(9, 6))
bars = plt.barh(brand_plot.index, brand_plot["margin_pct"], color=warna_brand)

for bar, (idx, row) in zip(bars, brand_plot.iterrows()):
    lebar = bar.get_width()
    tengah_y = bar.get_y() + bar.get_height() / 2
    plt.text(lebar + 0.5, tengah_y, f"{lebar:.1f}%", va="center", fontsize=10)
    if lebar > 5:
        plt.text(lebar / 2, tengah_y, f"n={int(row['n_transaksi'])}",
                 va="center", ha="center", fontsize=9, color="white")

plt.title("Margin Keuntungan per Brand", fontsize=13)
plt.xlabel("Margin (%)")
plt.ylabel("Brand")
plt.xlim(0, brand_plot["margin_pct"].max() + 8)
plt.tight_layout()
plt.show()

**Temuan:**

Converse (44,2%, n=4) dan Crocs (36,5%, n=4) muncul di posisi teratas, namun dengan hanya 4 transaksi masing-masing, margin ini tidak dapat diandalkan untuk keputusan strategis.

Perbandingan yang signifikan secara statistik adalah **Adidas vs Nike** — keduanya memiliki volume transaksi yang substansial:
- **Adidas: margin 35,1%** (n=123)
- **Nike: margin 22,8%** (n=662)

Meskipun Nike mewakili mayoritas transaksi (75% volume), **Adidas menghasilkan 12 poin persentase margin lebih tinggi per rupiah yang diinvestasikan.** Volume tinggi tidak berarti efisiensi tinggi.

**Rekomendasi:** Meningkatkan proporsi pembelian Adidas relatif terhadap Nike akan meningkatkan return keseluruhan. Namun ini harus diseimbangkan dengan ketersediaan pasar dan likuiditas — volume Nike yang lebih tinggi mungkin sebagian mencerminkan sell-through yang lebih cepat, yang juga mempengaruhi perputaran modal.

## 6. Analisis Dead Stock

Dead stock = barang yang sudah dibeli namun belum terjual. Modal yang diinvestasikan di sini secara efektif tertahan dan tidak tersedia untuk reinvestasi.

In [ ]:
# Modal yang tertahan di stok belum terjual
modal_tertahan = dead_stock["Modal"].sum()
total_modal_semua = df_all["Modal"].sum()
persen_modal = modal_tertahan / total_modal_semua * 100
persen_unit = len(dead_stock) / len(df_all) * 100

print(f"Total modal yang diinvestasikan (semua barang):  Rp {total_modal_semua:,.0f}")
print(f"Modal tertahan di dead stock:                   Rp {modal_tertahan:,.0f} ({persen_modal:.1f}% dari total modal)")
print(f"Unit tertahan di dead stock:                    {len(dead_stock)} unit ({persen_unit:.1f}% dari total unit)")
print(f"\n% Modal ({persen_modal:.1f}%) > % Unit ({persen_unit:.1f}%) — barang yang belum terjual cenderung lebih mahal dari rata-rata")

In [ ]:
# Pecah dead stock berdasarkan tahun pembelian
dead_stock = dead_stock.copy()
dead_stock["tahun_beli"] = dead_stock["MM/DD/YY (Beli)"].str[-4:]
dead_stock["tahun_beli"] = dead_stock["tahun_beli"].replace({"0222": "2022"})  # perbaiki typo input data

ringkasan_tahun = dead_stock.groupby("tahun_beli").agg(
    jumlah_unit=("Modal", "size"),
    total_modal=("Modal", "sum")
)
ringkasan_tahun

In [ ]:
warna = ["#4C72B0", "#4C72B0", "#C44E52", "#4C72B0"]  # sorot 2024 (konsentrasi terbesar)

plt.figure(figsize=(8, 5))
bar = plt.bar(ringkasan_tahun.index, ringkasan_tahun["total_modal"], color=warna)

for b in bar:
    tinggi = b.get_height()
    plt.text(b.get_x() + b.get_width()/2, tinggi,
             f"Rp {tinggi/1_000_000:.0f} jt",
             ha="center", va="bottom", fontsize=10)

plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x/1_000_000:.0f} jt")
)

plt.title("Modal Tertahan di Dead Stock per Tahun Pembelian", fontsize=13)
plt.xlabel("Tahun Pembelian")
plt.ylabel("Total Modal (Rp)")
plt.tight_layout()
plt.show()

**Temuan:**

Per pertengahan 2026, **seluruh 159 barang yang belum terjual sudah nyangkut minimal 1 tahun** — tidak ada yang bisa disebut "stok segar." Perinciannya:

| Tahun Beli | Unit | Modal Tertahan | Lama Nyangkut |
|---|---|---|---|
| 2022 | 46 | Rp 96,8 jt | ~3,5 tahun |
| 2023 | 20 | Rp 41,8 jt | ~2,5 tahun |
| 2024 | 92 | Rp 149,0 jt | ~1,5 tahun |
| 2025 | 1 | Rp 1,1 jt | ~1 tahun |

**Kelompok 2024 adalah perhatian paling mendesak** — 92 unit senilai Rp 149 jt (52% dari total modal beku) yang dibeli dalam satu tahun namun gagal terjual. Ini mengindikasikan potensi kesalahan keputusan pembelian di tahun tersebut dan perlu ditinjau ulang secara mendalam.

**Rekomendasi:** Prioritaskan likuidasi kelompok 2022 terlebih dahulu (paling lama disimpan, opportunity cost tertinggi) meskipun dengan margin yang dikurangi. Modal yang dibebaskan dari dead stock dapat digunakan kembali ke channel bermargian tinggi (Alias) dan brand efisien (Adidas).

---

## 7. Kesimpulan & Rekomendasi Strategis

| # | Temuan | Rekomendasi |
|---|---|---|
| 1 | Alias margin (28,5%) mengungguli Kick Avenue (22,0%) pada volume sebanding | Alihkan alokasi modal ke Alias |
| 2 | Adidas margin (35,1%) jauh melampaui Nike (22,8%) — Nike 5x lebih banyak transaksi | Tingkatkan proporsi pembelian Adidas |
| 3 | Rp 288,7 jt (16,5% modal) terkunci di dead stock, semua >1 tahun | Mulai likuidasi terstruktur, prioritaskan kelompok 2022 |
| 4 | Pembelian 2024 menghasilkan konsentrasi dead stock terbesar (Rp 149 jt) | Evaluasi kriteria pembelian 2024 sebelum scaling |

**Keterbatasan Analisis:**
- Novelship (n=2) dan Ox Street (n=5): margin tidak dapat diandalkan karena sampel sangat kecil
- Tidak ada data pelanggan (model marketplace) — analisis RFM/segmentasi pelanggan tidak dapat dilakukan
- Kolom tanggal memiliki kualitas input tidak konsisten — analisis holding period tidak dilakukan
- 21 item raffle (modal=0) dikecualikan dari perhitungan margin namun tetap dihitung dalam total profit
- Analisis hanya mencakup transaksi tertutup — tren harga pasar tidak dimasukkan